# COMP5329 — Week 11 Self-Study

# Self-Supervised Representation Learning: From Autoencoders to CLIP

**Semester 1, 2026**

---

## How to use this material

This notebook is a **standalone, comprehensive companion** to the Week 11 lecture. It is *not* an addendum to the 60-minute in-class tutorial — it is an independent document that covers **every concept on the lecture slides** in depth, with mathematical derivations and (where useful) short PyTorch snippets that illustrate the engineering view.

| Document | Purpose | Time budget |
|---|---|---|
| `Week 11-Self-Supervised Representation Learning.pptx` | Lecture slides (ground truth for content scope) | 90 min lecture |
| `Week11_Self_Supervised_Learning.ipynb` | In-class tutorial (review + practice + exam-style Qs) | 60 min tutorial |
| **`Week11_Self_Study_SSL.ipynb`** (this file) | **Full self-study** — covers everything on the slides, even where the tutorial also touches it | Self-paced, 3–5 hours |

If a topic appears in *both* the tutorial and here, that is **on purpose**: the tutorial is optimised for a 60-minute live session, while this document is optimised for deep understanding at your own pace. When the two overlap, this document goes further.

## Prerequisites

- **Week 5** — Convolutional networks (you'll see CNN encoders inside SimCLR / MoCo / BYOL).
- **Week 7** — Transformers and self-attention (you'll need ViT to read MAE and DINO).
- **Week 8** — BERT and Masked Language Modelling (MAE is "BERT for images" — we'll lean on the analogy heavily).
- **Week 9** — CLIP at the multimodal-foundation-model level. Here we revisit it through the **SSL lens**: CLIP = cross-modal InfoNCE.

## Roadmap

**Part I — Foundations.** Chapters 1–3 motivate self-supervision: why labelled data is the bottleneck, what humans do instead, how SSL turns the *data itself* into a supervisory signal, and the canonical generative-vs-discriminative split.

**Part II — Generative SSL.** Chapters 4–7 build the autoencoder family: classical AE (with the linear-AE ↔ PCA equivalence), denoising AE, why the AE programme plateaued, and the Masked Autoencoder (MAE) that revived it for ViTs.

**Part III — Contrastive learning.** Chapters 8–11 build contrastive SSL from first principles: positive/negative pairs, the InfoNCE loss (intuition, geometry, gradient analysis), SimCLR, and MoCo's queue + momentum encoder.

**Part IV — Non-contrastive learning.** Chapters 12–16 ask "do we even need negatives?": the *collapse* problem, the unifying *asymmetry principle*, BYOL, SimSiam, and DINO.

**Part V — Multimodal SSL.** Chapters 17–19 cover CLIP as cross-modal contrastive learning, its impact, its failure modes, and the philosophical contrast between SimCLR / DINO / CLIP.

**Part VI — Synthesis.** Chapter 20 distils the design space: every SSL method = pretext task + invariance choice + anti-collapse mechanism.


In [ ]:
# ── Imports used throughout this notebook ──────────────────────────────────
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device = {device}")


---

# Part I — Foundations

## Chapter 1 — Why Self-Supervised Learning?

*(Lecture slides 2–3)*

### 1.1 Models are limited by labels, not by data

Modern deep learning thrives on scale. ResNets, ViTs, and large language models all share the same recipe: a flexible architecture + a huge dataset + supervised cross-entropy. But that recipe has a hidden bottleneck — the **labels**, not the data.

| Resource | Status today |
|---|---|
| Raw images on the internet | $\sim 10^{12}$ and growing daily |
| Raw text on the internet | $\sim 10^{13}$ tokens crawled |
| Manually labelled images (ImageNet-21k) | $\sim 14 \times 10^6$ |
| Manually labelled segmentation masks (COCO) | $\sim 10^5$ |

Labels are:

- **Expensive** — a high-quality medical-imaging label can cost \$5–\$50 per example.
- **Limited in scale** — even Google's JFT-300M required dedicated infrastructure and is private.
- **Often noisy or biased** — annotators disagree, ontologies leak, popular categories dominate.
- **Brittle to distribution shift** — labels for "cars in California" don't transfer to "cars in Lagos".

The provocation behind self-supervised learning is simple: *if the data is the part of the pipeline that scales, design a learning signal that uses only the data.*

### 1.2 Are we fully using the data?

A standard supervised classifier sees image $x$ together with a one-hot label $y \in \{0, 1\}^C$. Information-theoretically, the label contributes $\log_2 C$ bits per example — for ImageNet-1k, that is **9.97 bits**. The image itself, even at modest 224×224×3 resolution, contains on the order of $\sim 10^5$ informative bits about textures, edges, parts, geometry, context, lighting, occlusion, and inter-object relationships.

> Supervised learning compresses an image to one of a thousand discrete buckets, and then asks a 25-million-parameter ResNet to match the bucket. The *bucket* is what the model is trying to predict; **everything else in the image is thrown away as noise**.

Self-supervised learning starts from the opposite stance: most of the structure of the world is already *in* the image. Spatial layout, object co-occurrence, multi-view consistency, and predictable continuations are all "free" learning signals — we just have to design a task that forces the network to extract them.

### 1.3 The one sentence definition

> **Self-supervised learning** is supervised learning on a task whose labels are *constructed from the input itself*, with no human annotation.

Every method in this notebook is a different answer to a single design question:

> *What part of the input should the network predict, and from what other part?*


## Chapter 2 — Learning without labels

*(Lecture slides 4–5)*

### 2.1 The human analogy

A child does not learn "dog" by reading $10^4$ labelled photographs. The child learns by:

1. **Observation** — watching a single dog from many angles, in many lights, doing many things.
2. **Prediction** — anticipating that the dog will bark, walk, eat, sleep.
3. **Completion** — recognising the dog when half of it is hidden behind a couch.
4. **Comparison** — telling the dog apart from a cat without a teacher labelling either.

Every one of these four cognitive operations has a direct analogue in self-supervised learning, and the next four parts of this notebook will mirror them:

| Cognitive operation | Mechanism | Method family |
|---|---|---|
| **Completion** — fill in what's missing | Reconstruction loss | AE, MAE (Part II) |
| **Comparison** — same vs different | Contrastive loss | InfoNCE, SimCLR, MoCo (Part III) |
| **Prediction** — anticipate the other view | Predictive alignment | BYOL, SimSiam, DINO (Part IV) |
| **Cross-modal binding** — image ↔ word | Cross-modal contrastive | CLIP (Part V) |

### 2.2 Pretext tasks

The mechanism that turns "no labels" into "supervised learning" is the **pretext task** — an artificial supervised problem whose targets are derived algorithmically from the input. Some historical pretext tasks (worth knowing, even though they have all been superseded):

- **Image colourisation**: input is a grayscale image, target is its colour version.
- **Jigsaw puzzles**: shuffle 9 patches of an image and predict the permutation.
- **Rotation prediction**: rotate the input by 0°/90°/180°/270° and predict which.
- **Relative patch position**: crop two patches and predict their relative spatial offset.
- **Inpainting**: erase a square and reconstruct it.

All of these gave the network a free supervised problem. The lessons we keep from them are:

1. **Pretext tasks transfer** — a network pre-trained on rotation prediction is a meaningfully better initialisation for ImageNet classification than random.
2. **The choice of pretext task determines what the network learns** — rotation prediction can be solved by detecting horizons, so the network learns "horizon-ish" features and not much else.
3. **Better pretext tasks → better features**, and the rest of this course is about modern, much stronger pretext tasks.

The methods we study from Chapter 4 onward are *not* a different paradigm from rotation/jigsaw — they are simply much more carefully designed pretext tasks whose solutions require *general-purpose visual understanding*.


## Chapter 3 — The SSL landscape: generative vs discriminative

*(Lecture slide 6)*

There are two large families of self-supervised methods, separated by **where the loss lives**.

### 3.1 Generative SSL

> **Predict the missing or corrupted part of the input, in input space.**

- **Loss space**: pixels, patches, or tokens — i.e. the same space as the raw input.
- **Architecture**: encoder → latent → **decoder** → reconstruction.
- **Examples**: PCA, AE, denoising AE, VAE, MAE, BERT MLM, GPT next-token prediction.
- **Strength**: forces the latent to retain **enough information** to rebuild the input. Strong information-theoretic guarantee.
- **Weakness**: pixel-space losses overweight low-frequency texture and lighting; the decoder absorbs capacity that we wanted in the encoder; the encoder learns no explicit notion of *invariance*.

### 3.2 Discriminative SSL

> **Match representations of two related views, in embedding space.**

- **Loss space**: an internal embedding space (typically L2-normalised, 128–4096-dim).
- **Architecture**: two encoders (often weight-shared), no decoder.
- **Examples**:
  - *Contrastive* — needs negatives: SimCLR, MoCo, CLIP.
  - *Non-contrastive* — no negatives, anti-collapse comes from architecture: BYOL, SimSiam, DINO.
- **Strength**: directly optimises representation **invariance** to chosen augmentations.
- **Weakness**: depends critically on the augmentation distribution; vulnerable to *representation collapse* if the loss is misdesigned.

### 3.3 The unified slogan

Through both families runs one principle:

> **Every SSL method = (pretext task that defines what should be the same) + (anti-collapse mechanism that prevents the trivial solution).**

The pretext task says *what invariance to learn*. The anti-collapse mechanism stops the network from satisfying that pretext task in a useless way (e.g., mapping every input to the constant zero vector).

Keep this slogan in mind. From Chapter 4 onward, every method we cover can be summarised in one row of a table whose columns are (1) what the supervision signal is, (2) what it is predicting, (3) how it avoids collapse.


---

# Part II — Generative Self-Supervision

## Chapter 4 — Autoencoders

*(Lecture slides 8–10)*

### 4.1 The simplest possible idea

> **The label IS the input itself.**

An autoencoder is a pair of networks $(f_\theta, g_\phi)$ that pass the input through a low-dimensional bottleneck and try to reconstruct it on the other side:

$$
\hat{x} = g_\phi(f_\theta(x)), \qquad
\mathcal{L}_{\text{AE}}(\theta, \phi) = \mathbb{E}_{x \sim \mathcal{D}} \|x - \hat{x}\|_2^2.
$$

The encoder $f_\theta : \mathbb{R}^D \to \mathbb{R}^d$ compresses; the decoder $g_\phi : \mathbb{R}^d \to \mathbb{R}^D$ reconstructs. The number $d$ is the **bottleneck dimension** and is the single most important hyper-parameter.

| Bottleneck $d$ | What happens |
|---|---|
| $d \ge D$ (overcomplete) | Network can learn the identity, $\hat{x} = x$, with zero training loss and useless features. **Needs regularisation.** |
| $d \approx D / 10$ (undercomplete) | Reconstruction is no longer free; the encoder must *prioritise* which information to keep. |
| $d \ll D$ (heavily undercomplete) | Strong compression; reconstructions blur. Often the best regime for representation learning. |

### 4.2 Linear AE = PCA

If both $f_\theta$ and $g_\phi$ are linear (i.e. $f_\theta(x) = W_e x$, $g_\phi(z) = W_d z$, no biases, no nonlinearity), and the loss is squared error, then the optimal $(W_e, W_d)$ recover the **top-$d$ principal components** of the data — exactly the same subspace as PCA.

*Sketch of proof.* For zero-mean data with covariance $\Sigma = \mathbb{E}[xx^\top]$, the loss

$$\mathcal{L}(W_e, W_d) = \mathbb{E}\,\|x - W_d W_e x\|_2^2 = \mathrm{tr}(\Sigma) - 2\,\mathrm{tr}(W_d W_e \Sigma) + \mathrm{tr}(W_e^\top W_d^\top W_d W_e \Sigma)$$

is minimised when $W_d W_e$ is the projection onto the subspace spanned by the top-$d$ eigenvectors of $\Sigma$. Any rotation inside that subspace is also optimal — i.e. the linear AE finds the *PCA subspace* but not necessarily the PCA basis.

This is an important sanity result: it tells us that the simplest possible AE is *exactly* a method we already understood. Nonlinear AEs generalise PCA to curved manifolds, and that is where the real power begins.

### 4.3 Denoising autoencoders (Vincent et al., 2008)

If the AE is allowed to learn the identity, it is useless. One way to break that escape route is **denoising**: corrupt the input first, then ask the network to recover the *clean* original.

$$
\tilde{x} = x + \epsilon, \qquad
\mathcal{L}_{\text{DAE}} = \mathbb{E}_{x, \epsilon}\,\|x - g_\phi(f_\theta(\tilde{x}))\|_2^2.
$$

Common corruptions: additive Gaussian noise, salt-and-pepper, random erasing, random patch masking. The intuition: identity is no longer an option, because $\tilde{x} \ne x$ — the network *has* to model the data manifold to know which direction to push $\tilde{x}$ back.

Denoising AEs were the first SSL method to give competitive transfer learning results on real image benchmarks, and they directly inspired both BERT (15% token corruption) and MAE (75% patch masking).

### 4.4 A working autoencoder on MNIST

Below we train a small fully-connected AE with a 32-d bottleneck on MNIST. Pay attention to the bottleneck — it is only 32 numbers, but they must be enough to reconstruct a 784-pixel digit.


In [ ]:
# ── Autoencoder from scratch on MNIST ───────────────────────────────────────
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Tiny MNIST loader (downloads ~12MB on first run)
tfm = transforms.ToTensor()
train_ds = datasets.MNIST("./data", train=True, download=True, transform=tfm)
loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)

class AE(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, d),
        )
        self.dec = nn.Sequential(
            nn.Linear(d, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )
    def forward(self, x):
        z = self.enc(x.view(x.size(0), -1))
        return self.dec(z), z

model = AE(d=32).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(2):  # short demo
    for x, _ in loader:
        x = x.to(device)
        x_hat, _ = model(x)
        loss = F.mse_loss(x_hat, x.view(x.size(0), -1))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch}: loss = {loss.item():.4f}")


In [ ]:
# ── Visualise reconstructions and latent code ──────────────────────────────
model.eval()
with torch.no_grad():
    x, _ = next(iter(loader))
    x = x[:8].to(device)
    x_hat, z = model(x)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(x[i, 0].cpu(), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i].view(28, 28).cpu(), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_title("original", loc="left")
axes[1, 0].set_title("reconstruction", loc="left")
plt.suptitle(f"32-d bottleneck encodes 784 pixels — first 4 latent dims of x[0]: {z[0,:4].cpu().numpy().round(2)}")
plt.tight_layout(); plt.show()


## Chapter 5 — Why the AE programme plateaued

*(Lecture slide 11)*

The autoencoder is conceptually beautiful and historically important (Hinton & Salakhutdinov, *Science* 2006), but by ~2015 the field had largely abandoned it as a representation-learning tool. The reasons are worth enumerating because each subsequent method (MAE, contrastive, non-contrastive) is best understood as a *fix* for one of these failure modes.

### 5.1 Pixels are not semantics

The loss $\|x - \hat{x}\|_2^2$ rewards getting *every pixel* right. But the pixel value at coordinate $(57, 102)$ has almost no semantic content on its own. A reconstruction that gets the cat's whisker exactly right but loses the cat-ness scores well; a reconstruction that captures the cat-ness but blurs the whiskers scores badly. **The loss is misaligned with the goal.**

### 5.2 MSE is dominated by high-frequency texture

L2 loss in pixel space is a low-pass filter on what the network cares about. A 1-pixel shift produces a huge MSE; a global change of object identity might produce a small one. The network's gradients are dominated by edges, not by parts.

### 5.3 The decoder steals capacity from the encoder

In a symmetric encoder–decoder, half the parameters are spent learning to **draw**. We don't want a drawing network — we want a representation network. The decoder is discarded after pre-training, so every parameter inside it is wasted from a downstream-task point of view.

### 5.4 No notion of invariance

Two augmented views of the same image — say a crop and a colour jitter — should map to similar representations if the goal is robust visual understanding. The AE objective never asks for this. It only asks: can you rebuild this exact image? Nothing prevents the encoder from learning a *brittle* code.

### 5.5 The three successors, in one table

Each of the three method families covered later directly fixes one of the above limitations:

| Successor | Fixes which limitation? | How |
|---|---|---|
| **MAE** | "Decoder steals capacity" | Make the decoder lightweight and the encoder process only visible patches. Reconstructs in pixel space but in a way that *forces the encoder to do the work.* |
| **Contrastive** (SimCLR, MoCo) | "No invariance" + "pixels ≠ semantics" | Replace pixel reconstruction with embedding-space comparison of two augmented views. |
| **Non-contrastive** (BYOL, SimSiam, DINO) | "Pixels ≠ semantics" | Same as contrastive, but remove the negatives and prevent collapse with architectural asymmetry. |

The AE is therefore not so much *wrong* as *incomplete*. It is the simplest member of a much richer family, and the rest of this notebook tells the story of how the family grew.


## Chapter 6 — Masked Autoencoders (MAE)

*(Lecture slides 12–16; He et al., CVPR 2022)*

### 6.1 The one-line idea

> Take a Vision Transformer, hide 75% of its input patches, and ask it to reconstruct the missing ones.

MAE is BERT for images: a masked-token reconstruction task on patch tokens instead of word tokens. But it makes one critical *architectural* change that BERT did not need — and that change is what makes MAE both more accurate and ~3× cheaper to train than its predecessors.

### 6.2 The pipeline

1. **Patchify**: split the image into non-overlapping $16 \times 16$ patches. A $224 \times 224$ image becomes $14 \times 14 = 196$ patch tokens.
2. **Random mask**: sample 75% of the patches uniformly at random and *delete them entirely*. The encoder will never see them.
3. **Encoder**: a heavy ViT (e.g. ViT-L: 24 blocks, 1024-d) processes the **remaining 25% of patches only**.
4. **Insert mask tokens**: at the decoder input, the missing positions are filled with a single learnable `[MASK]` embedding (plus position embedding).
5. **Decoder**: a *lightweight* ViT (8 blocks, 512-d) reconstructs every masked patch's pixels.
6. **Loss**: MSE on the **masked positions only**:

$$
\mathcal{L}_{\text{MAE}} = \frac{1}{|M|} \sum_{i \in M} \|p_i - \hat{p}_i\|_2^2.
$$

### 6.3 Why the asymmetric design is the key trick

The non-obvious design choice is that **mask tokens are not given to the encoder**. In BERT, they are — the encoder sees a mix of real tokens and `[MASK]` tokens. MAE breaks that convention, and the consequences are large:

| Consequence | Why it matters |
|---|---|
| **Encoder cost drops 4×** | Self-attention is $O(N^2)$ in token count. Passing only 25% of tokens cuts encoder FLOPs by $\sim (1/4)^2 = 1/16$ per attention layer — and the encoder is where almost all the FLOPs live. |
| **Encoder gets harder problem** | The encoder cannot "cheat" by attending to mask tokens that signal *where* the holes are. It must build a representation good enough that the decoder can later inpaint. |
| **Train/test distribution gap closes** | At downstream time the encoder sees no mask tokens, so it never learns to depend on them. |
| **Decoder can be tiny** | Reconstruction is still happening, but the decoder is discarded after pre-training. We *want* it small. |

### 6.4 Why 75% masking? (And why not BERT's 15%?)

| Mask ratio | Behaviour |
|---|---|
| **15% (BERT-like)** | Task is trivial — you can recover most patches by interpolating their immediate neighbours. The encoder learns local texture, not global structure. |
| **75% (MAE optimum)** | The remaining 25% of patches are spatially sparse. To reconstruct, the encoder must reason about *whole-image structure* (where the dog's head is, what direction it faces, what fills the background). |
| **95%** | Too few visible patches to disambiguate. Reconstructions hallucinate; representation quality drops. |

The reason images can tolerate (and benefit from) such heavy masking is **spatial redundancy**: adjacent pixels are highly correlated. A single dog patch is *highly informative* about its neighbours. Language is information-dense — a single token may carry an entire concept ("Bayesian"), so masking 75% of words leaves an unrecoverable hole.

This is the deepest lesson of MAE: **the right mask ratio is determined by the redundancy of the modality.** Audio has high temporal redundancy, so AudioMAE works well at ~80%. Video has even higher redundancy, so VideoMAE pushes to 90%.

### 6.5 A toy MAE-style masking demonstration

We don't have ViT-Huge or ImageNet here, but the *masking + reconstruct-only-the-masked-positions* idea is small enough to demonstrate on MNIST patches. Below: split each digit into $4 \times 4$ patches (49 patches), randomly mask 75% of them, and train a tiny model to reconstruct. The point is to feel the difficulty, not to set a benchmark.


In [ ]:
# ── Toy MAE: 4x4 patches on MNIST, 75% masking ─────────────────────────────
PATCH = 4
NPATCH = (28 // PATCH) ** 2   # 49 patches per image

def patchify(x):  # x: (B, 1, 28, 28) -> (B, 49, 16)
    B = x.size(0)
    p = x.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH)
    return p.contiguous().view(B, NPATCH, PATCH * PATCH)

def unpatchify(p):  # (B, 49, 16) -> (B, 1, 28, 28)
    B = p.size(0)
    side = 28 // PATCH
    p = p.view(B, side, side, PATCH, PATCH)
    p = p.permute(0, 1, 3, 2, 4).contiguous()
    return p.view(B, 1, 28, 28)

class TinyMAE(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.embed = nn.Linear(PATCH * PATCH, d)
        self.pos   = nn.Parameter(torch.zeros(1, NPATCH, d))
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 4, 256, batch_first=True), num_layers=4)
        self.mask_tok = nn.Parameter(torch.zeros(1, 1, d))
        self.decoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 4, 256, batch_first=True), num_layers=2)
        self.out = nn.Linear(d, PATCH * PATCH)

    def forward(self, x, mask_ratio=0.75):
        B = x.size(0)
        patches = patchify(x)                  # (B, 49, 16)
        tokens  = self.embed(patches) + self.pos  # (B, 49, d)

        # Random mask: pick 25% to keep
        n_keep = int(NPATCH * (1 - mask_ratio))
        noise = torch.rand(B, NPATCH, device=x.device)
        ids_shuffle = noise.argsort(dim=1)     # (B, 49) random order
        ids_keep    = ids_shuffle[:, :n_keep]
        ids_restore = ids_shuffle.argsort(dim=1)

        idx = ids_keep.unsqueeze(-1).expand(-1, -1, tokens.size(-1))
        visible = torch.gather(tokens, 1, idx)  # encoder sees only these

        # ---- Encoder: visible patches only (the asymmetry trick) ----
        enc = self.encoder(visible)             # (B, 12, d)

        # ---- Decoder: insert mask tokens at the holes ----
        n_mask = NPATCH - n_keep
        mask_tokens = self.mask_tok.expand(B, n_mask, -1)
        full = torch.cat([enc, mask_tokens], dim=1)  # (B, 49, d) but in shuffled order
        idx_unshuffle = ids_restore.unsqueeze(-1).expand(-1, -1, full.size(-1))
        full = torch.gather(full, 1, idx_unshuffle)
        full = full + self.pos

        dec  = self.decoder(full)
        recon_patches = self.out(dec)           # (B, 49, 16)

        # Loss only on masked positions
        target = patches
        mask = torch.ones(B, NPATCH, device=x.device)
        mask.scatter_(1, ids_keep, 0.0)         # 1 = masked, 0 = visible
        loss = ((recon_patches - target) ** 2).mean(-1)  # (B, 49)
        loss = (loss * mask).sum() / mask.sum()

        return loss, unpatchify(recon_patches), mask

model = TinyMAE().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(2):
    for x, _ in loader:
        x = x.to(device)
        loss, _, _ = model(x)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch}: masked-MSE = {loss.item():.4f}")


In [ ]:
# ── Visualise: original | masked | reconstructed ──────────────────────────
model.eval()
with torch.no_grad():
    x, _ = next(iter(loader))
    x = x[:6].to(device)
    _, recon, mask = model(x)

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for i in range(6):
    axes[0, i].imshow(x[i, 0].cpu(), cmap="gray"); axes[0, i].axis("off")
    # masked input
    masked_img = patchify(x)[i].clone()
    masked_img[mask[i].bool()] = 0
    axes[1, i].imshow(unpatchify(masked_img.unsqueeze(0))[0,0].cpu(), cmap="gray")
    axes[1, i].axis("off")
    axes[2, i].imshow(recon[i, 0].cpu().clamp(0, 1), cmap="gray"); axes[2, i].axis("off")
axes[0, 0].set_title("original", loc="left")
axes[1, 0].set_title("75% masked", loc="left")
axes[2, 0].set_title("reconstruction", loc="left")
plt.tight_layout(); plt.show()


## Chapter 7 — MAE vs Classic AE

*(Lecture slides 17–19)*

It is tempting to think of MAE as "an autoencoder with masking". This is misleading. They are *fundamentally different pretext tasks* and, after the first sentence, share almost nothing.

| Aspect | Classic AE | MAE |
|---|---|---|
| Pretext task | Reconstruct entire input | Reconstruct only the **masked 75%** |
| Difficulty | Trivial — input is fully visible | Hard — must infer missing structure from sparse evidence |
| Loss target | All pixels | Masked pixels only |
| Encoder input | Whole image | **25% of patches only** |
| Encoder/decoder balance | Symmetric | Heavy encoder, **lightweight** decoder |
| Bottleneck | A small latent code $z$ | No explicit bottleneck — the *masking* is the bottleneck |
| Anti-collapse mechanism | Bottleneck dimension $d \ll D$ | High mask ratio + reconstruction-from-context |
| Inference: what you keep | Encoder + decoder | **Encoder only** (decoder discarded) |

A useful mental model:

- A classic AE has a **dimensional** bottleneck — the latent vector $z$ is forced to be low-rank.
- An MAE has a **spatial** bottleneck — the visible patches are forced to cover only 25% of the image.

Both create scarcity. The dimensional kind discards information *globally* (every pixel is summarised); the spatial kind discards information *locally* (some regions are intact, others are gone). Empirically, the spatial form turns out to be much better, because it forces the encoder to develop something closer to *object-level reasoning* — the only way to fill in a missing patch is to know what is going on around it.

### Limitations of MAE

MAE is the strongest member of the generative family, but it inherits the original AE's deepest problem: **it still optimises in pixel space**. Reconstructing pixels does not directly optimise *invariance* — two crops of the same dog should plausibly map to similar representations, but MAE never asks for that. Empirically, MAE features are *excellent* for downstream tasks that need fine spatial detail (detection, segmentation), and *somewhat weaker* than contrastive features on linear-probe ImageNet classification. The next part of the course explains why.


---

# Part III — Contrastive Learning

## Chapter 8 — From generative to discriminative

*(Lecture slides 20–22)*

### 8.1 The pivot

Generative SSL says: *predict pixels*. Contrastive SSL says: *predict similarity*.

There is no reconstruction. There is no decoder. There are only an **encoder** and an **embedding space** in which we declare:

- **Two views of the same image** should be **close**.
- **Views of different images** should be **far apart**.

That is the whole idea. Everything else — InfoNCE, SimCLR's huge batches, MoCo's queue — is a way to make this idea work in practice.

### 8.2 Positive and negative pairs

Given a minibatch of $N$ images, the contrastive recipe constructs:

- **Positive pair**: two random augmentations of the *same* image,
  $$v = t(x), \quad v' = t'(x), \quad t, t' \sim \mathcal{T}$$
  where $\mathcal{T}$ is a distribution over augmentations (random crop, colour jitter, blur, flip, …).
- **Negatives**: views of all *other* images in the same batch.

The augmentation distribution $\mathcal{T}$ is the unsung hero of contrastive learning. It encodes the **invariances we want the model to learn**:

| Augmentation | Invariance taught |
|---|---|
| Random resized crop | Spatial location, partial visibility, scale |
| Colour jitter | Lighting, white balance |
| Random gray | Colour identity |
| Gaussian blur | High-frequency texture |
| Horizontal flip | Left/right symmetry of natural images |
| (No vertical flip) | Up/down is *not* an invariance — gravity matters |

Choose the augmentations and you have chosen the geometry of the embedding space. SimCLR's most quoted ablation (Chen et al., 2020, Table 1) shows that **colour distortion is the single most important augmentation** — without it, the network solves the task by reading colour histograms.

### 8.3 What we want, in one sentence

> A representation $f$ such that **for any image** $x$ and **any pair of augmentations** $(t, t')$, $f(t(x))$ and $f(t'(x))$ are closer than $f(t(x))$ and $f(t(y))$ for any other image $y$.

This is the property called **alignment + uniformity** (Wang & Isola, 2020):

- **Alignment** — augmented views of the same image map to nearby embeddings.
- **Uniformity** — embeddings of different images are spread uniformly on the unit hypersphere.

InfoNCE optimises both at once. The next chapter shows how.


## Chapter 9 — InfoNCE: the workhorse loss

*(Lecture slides 23–27; van den Oord et al., 2018)*

### 9.1 Intuition: instance discrimination as $(K{+}1)$-way classification

Take an anchor image and produce its embedding $q$. Take its positive (a different augmentation of the same image) and produce $k^+$. Take $K$ other images and produce $\{k^-_1, \ldots, k^-_K\}$. Now ask the network:

> *"Out of these $K + 1$ candidate keys, which one is the positive?"*

This is a $(K+1)$-way classification problem with one correct answer. The "logits" are similarities $\mathrm{sim}(q, k_j) / \tau$, and the cross-entropy loss is:

$$
\boxed{\;\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\mathrm{sim}(q, k^+) / \tau)}{\exp(\mathrm{sim}(q, k^+) / \tau) + \sum_{j=1}^{K}\exp(\mathrm{sim}(q, k^-_j) / \tau)}\;}
$$

This view of InfoNCE — "softmax over candidates" — is the most useful one. Every contrastive method we cover (SimCLR, MoCo, CLIP) is a different choice of *where the positives and negatives come from*; the loss itself is identical.

### 9.2 Information-theoretic background

InfoNCE is named "Noise-Contrastive Estimation" and is a **lower bound on the mutual information** between $q$ and $k^+$:

$$
I(q; k^+) \;\geq\; \log K - \mathcal{L}_{\text{InfoNCE}}.
$$

Two consequences:

1. Minimising the loss maximises (a lower bound on) the mutual information between two views.
2. **More negatives = tighter bound.** This is the entire reason SimCLR uses batch sizes of 4096 and CLIP uses 32768: more negatives → better InfoNCE → better representations.

### 9.3 Temperature, in three regimes

The temperature $\tau$ scales the logits. Its role is more important than it appears.

| Regime | Softmax behaviour | Effect on training |
|---|---|---|
| $\tau \to 0$ | Winner-take-all; softmax becomes argmax | Gradient concentrates entirely on the *single hardest* negative. Training is unstable. |
| $\tau$ "right" (≈ 0.07–0.5) | Soft but peaked | Gradient distributes across hard negatives proportionally to difficulty. |
| $\tau \to \infty$ | Uniform | All negatives weighted equally; the loss saturates and provides almost no gradient signal. |

Practical values: SimCLR uses $\tau = 0.5$ on ImageNet ($\tau = 0.1$ on CIFAR); MoCo uses $\tau = 0.07$; CLIP **learns** $\tau$ as a parameter, initialised so that $1/\tau \approx 14$ (i.e. $\tau \approx 0.07$) and clipped to prevent divergence.

### 9.4 Geometric interpretation

After projection, embeddings are **L2-normalised**: $\|q\| = \|k\| = 1$. They live on the unit hypersphere $S^{d-1}$. Cosine similarity becomes the dot product:

$$\mathrm{sim}(q, k) = q^\top k = \cos\angle(q, k).$$

In this picture:

- **Pulling positives together** ↔ rotating $q$ toward $k^+$ on the sphere.
- **Pushing negatives apart** ↔ rotating $q$ away from each $k^-_j$.

The unit sphere is the right geometry because (a) it removes magnitude as a degree of freedom (the network can't cheat by making vectors longer), and (b) it makes cosine similarity bounded in $[-1, 1]$, so temperature has a stable interpretation.

### 9.5 The InfoNCE gradient: pull and push, weighted by hardness

This is the mathematically beautiful part of InfoNCE — the gradient *automatically* focuses on hard negatives. Let $s_+ = q^\top k^+ / \tau$ and $s_j = q^\top k^-_j / \tau$. Define the softmax probabilities:

$$
p_+ = \frac{e^{s_+}}{e^{s_+} + \sum_j e^{s_j}}, \qquad
p_j = \frac{e^{s_j}}{e^{s_+} + \sum_j e^{s_j}}.
$$

Then a short calculation gives

$$
-\nabla_q \mathcal{L}_{\text{InfoNCE}}
= \frac{1}{\tau}\Big[(1 - p_+)\, k^+ \;-\; \sum_j p_j\, k^-_j\Big].
$$

Two forces acting on $q$:

1. **Pull** toward the positive $k^+$, scaled by $(1 - p_+)$ — i.e., scaled by *how badly we are currently doing* on the positive.
2. **Push** away from a *weighted average* of the negatives, where each negative's weight is its softmax probability $p_j$.

The weights $p_j$ are concentrated on the **hard negatives** — the ones whose similarity $s_j$ is largest. Easy negatives (already far from $q$) get tiny weights and contribute almost no gradient. **InfoNCE does adaptive hard-negative mining for free.**

This is one of the reasons contrastive methods are so robust: the curriculum is automatic. Early in training, almost all negatives are "hard" (the encoder is random); late in training, only a few subtle negatives remain hard, and those get all the gradient.

### 9.6 InfoNCE from scratch


In [ ]:
# ── InfoNCE from scratch ────────────────────────────────────────────────────
def info_nce(q, k_pos, k_neg, tau=0.07):
    # q     : (B, d)   anchor embeddings (already L2-normalised)
    # k_pos : (B, d)   positive keys
    # k_neg : (K, d)   negative keys (shared across batch)
    # Logits: positive (B,1) and negatives (B,K)
    pos = (q * k_pos).sum(-1, keepdim=True) / tau          # (B, 1)
    neg = q @ k_neg.t() / tau                              # (B, K)
    logits = torch.cat([pos, neg], dim=1)                  # (B, 1+K)
    labels = torch.zeros(q.size(0), dtype=torch.long, device=q.device)  # positive at index 0
    return F.cross_entropy(logits, labels)

# Sanity check
B, K, d = 32, 256, 64
q = F.normalize(torch.randn(B, d), dim=-1)
kp = F.normalize(q + 0.05 * torch.randn(B, d), dim=-1)  # near-positives
kn = F.normalize(torch.randn(K, d), dim=-1)
print(f"InfoNCE (B={B}, K={K}, tau=0.07) = {info_nce(q, kp, kn).item():.4f}")
print(f"Random baseline (log(K+1))      = {math.log(K + 1):.4f}")


In [ ]:
# ── Empirical demonstration: gradient concentrates on hard negatives ───────
torch.manual_seed(7)
d = 32
q = F.normalize(torch.randn(1, d, requires_grad=True), dim=-1)
k_pos = F.normalize(torch.randn(1, d), dim=-1)

# 19 random "easy" negatives + 1 deliberately hard one (close to q)
k_neg_easy = F.normalize(torch.randn(19, d), dim=-1)
k_neg_hard = F.normalize(q.detach() + 0.1 * torch.randn(1, d), dim=-1)
k_neg = torch.cat([k_neg_easy, k_neg_hard], dim=0)

loss = info_nce(q, k_pos, k_neg, tau=0.1)
loss.backward()

# Inspect the softmax weights to see which negative the gradient targets
with torch.no_grad():
    pos_logit = (q * k_pos).sum() / 0.1
    neg_logits = (q @ k_neg.t()).flatten() / 0.1
    all_logits = torch.cat([pos_logit.view(1), neg_logits])
    probs = F.softmax(all_logits, dim=0)
    print(f"Softmax weight on positive  = {probs[0].item():.3f}")
    print(f"Mean weight on 19 easy negs = {probs[1:20].mean().item():.4f}")
    print(f"Weight on the 1 hard neg    = {probs[20].item():.4f}  ← gradient sink")


## Chapter 10 — SimCLR

*(Lecture slides 28–29; Chen et al., ICML 2020)*

### 10.1 The architecture, end to end

SimCLR = "InfoNCE applied with the simplest possible plumbing".

1. Sample a minibatch of $N$ images.
2. Apply two random augmentations to each, producing $2N$ views.
3. Pass each view through a backbone $f$ (e.g. ResNet-50): $h_i = f(\tilde x_i)$.
4. Project to a small unit-sphere embedding via a 2-layer MLP $g$: $z_i = \mathrm{normalize}(g(h_i))$.
5. For each view, the *other view of the same image* is the positive; the other $2(N-1)$ views are negatives.
6. Loss is a symmetric InfoNCE (NT-Xent) summed over all $2N$ anchors.

There is no memory bank, no momentum encoder, no separate target network. Just a backbone + projector + contrastive loss + a *huge* batch size.

### 10.2 The three things that actually matter

#### (a) The augmentation composition

This is by far the most important hyper-parameter. The ablation table in Chen et al. (2020, Fig. 5) shows that no single augmentation is enough — the magic is the *composition* (especially crop + colour distortion). SimCLR's augmentation pipeline:

```
RandomResizedCrop(224, scale=(0.08, 1.0))
RandomHorizontalFlip()
ColorJitter(0.4, 0.4, 0.4, 0.1) with prob 0.8
RandomGrayscale(prob=0.2)
GaussianBlur(kernel=23) with prob 0.5
```

Removing colour jitter alone drops linear-probe accuracy from 69% to 60%. Removing crop alone drops it below 50%.

#### (b) The projection head

The forward path is

$$x \xrightarrow{f} h \in \mathbb{R}^{2048} \xrightarrow{g} z \in \mathbb{R}^{128}.$$

An empirical surprise: **the representation used for downstream tasks is $h$, not $z$**. The projection head $g$ is *thrown away* after pre-training.

Why? The contrastive loss in $z$-space pushes the representation to be **invariant** to the augmentations. Some of those invariances are useful (lighting); some destroy information that downstream tasks need (e.g. colour, which would be needed for bird-species classification). The head $g$ gives the model a "scratch space" in which it can be aggressively invariant for the contrastive task while letting $h$ keep more information.

This finding generalises: **always evaluate the layer before the projection head.**

#### (c) Scale

Bigger batch → more negatives → tighter InfoNCE bound → better representation. SimCLR's scaling curve is monotonic up to batch 8192.

### 10.3 Strengths and limitations

**Strengths.**
- The simplest possible contrastive recipe — clean ablation platform.
- A strong baseline: 69.3% linear probe with ResNet-50 (batch 4096, 1000 epochs).
- No memory bank, no momentum encoder, no extra hyper-parameters.

**Limitations.**
- Wall-clock cost dominated by the **enormous batches** required.
- $O(N^2)$ similarity matrix at every step.
- Practically out of reach for most academic labs (8 V100s minimum to train at full scale).

The next chapter — MoCo — asks the obvious question: *can we keep the negatives without keeping the giant batch?*

### 10.4 NT-Xent (SimCLR's exact form of InfoNCE) from scratch


In [ ]:
# ── NT-Xent: the SimCLR loss, from scratch ─────────────────────────────────
def nt_xent(z, tau=0.5):
    # z : (2N, d) batch of L2-normalised embeddings.
    # Row i holds view-1 of image i; row N + i holds view-2 of the same image.
    # Returns the symmetric NT-Xent loss summed over all 2N anchors.
    twoN = z.size(0)
    N = twoN // 2

    sim = (z @ z.t()) / tau                           # (2N, 2N)
    # mask out self-similarity on the diagonal
    mask = torch.eye(twoN, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, float("-inf"))

    # The positive of row i is row (i + N) mod 2N
    targets = torch.arange(twoN, device=z.device)
    targets = (targets + N) % twoN
    return F.cross_entropy(sim, targets)

# Sanity check
N, d = 16, 32
z1 = F.normalize(torch.randn(N, d), dim=-1)
z2 = F.normalize(z1 + 0.1 * torch.randn(N, d), dim=-1)  # positive views
z  = torch.cat([z1, z2], dim=0)
print(f"NT-Xent (N={N}, tau=0.5) = {nt_xent(z).item():.4f}")
print(f"Random  baseline log(2N-1) = {math.log(2*N - 1):.4f}")


## Chapter 11 — MoCo: Momentum Contrast

*(Lecture slides 30–31; He et al., CVPR 2020)*

### 11.1 The problem MoCo solves

SimCLR's recipe is: more negatives → better. But "more negatives" in SimCLR means "bigger batch", which means "more GPUs". Can we get tens of thousands of negatives from a *normal* batch?

MoCo's answer is: *yes, by storing recently encoded negatives in a queue.* But that creates a new problem — if the encoder updates every step, the keys in the queue go stale immediately. MoCo's fix is the **momentum encoder**: a slowly evolving copy of the main encoder so that all keys in the queue come from "almost the same" network.

### 11.2 The architecture

There are two encoders:

- **Query encoder** $f_q$ — a normal encoder, updated by gradient descent.
- **Key encoder** $f_k$ — an exponential moving average (EMA) of $f_q$:
  $$\theta_k \leftarrow m\, \theta_k + (1 - m)\, \theta_q, \qquad m = 0.999.$$
  No gradients flow through $f_k$.

For each input $x$, we apply two augmentations: $f_q$ encodes one view as a query $q$, $f_k$ encodes the other as the positive key $k^+$.

The negatives come from a **queue** $\mathcal{Q}$ of keys produced by $f_k$ on *previous batches*. The queue is FIFO with capacity $K$ (e.g. $K = 65536$); each step we enqueue the new keys and dequeue the oldest.

The loss is plain InfoNCE:

$$\mathcal{L}_{\text{MoCo}} = -\log \frac{\exp(q^\top k^+ / \tau)}{\exp(q^\top k^+ / \tau) + \sum_{k \in \mathcal{Q}} \exp(q^\top k / \tau)}.$$

### 11.3 Why momentum is essential

Consider what would happen *without* momentum — i.e. if we used the live $f_q$ to compute the queue keys.

- Step $t$: encode batch with $f_q^{(t)}$, push keys into queue.
- Step $t + 1$: $f_q$ has updated to $f_q^{(t+1)}$. The queue still holds keys from $f_q^{(t)}$, $f_q^{(t-1)}$, etc.
- The query at step $t+1$ is in the geometry of $f_q^{(t+1)}$, but it is being compared to keys from many older networks. The comparison is **inconsistent**.

With momentum $m = 0.999$:

- Per step, $f_k$ moves only $0.1\%$ of the way toward $f_q$.
- Over the lifetime of the queue (≈ 256 steps for batch 256, $K = 65536$), $f_k$ has changed by at most $\sim 25\%$ of the gap.
- All keys in the queue are "almost from the same encoder". Comparison is consistent.

Mathematically, the EMA is a low-pass filter on the parameter trajectory. The queue is then a low-pass-filtered snapshot of the encoder over a recent window of training.

### 11.4 SimCLR vs MoCo: head-to-head

| Aspect | SimCLR | MoCo |
|---|---|---|
| Source of negatives | Current batch ($2(N-1)$) | Queue ($K = 65536$) |
| Required batch size | 4096–8192 (huge) | 256 (normal) |
| Key encoder | Same as query (shared weights) | EMA of query |
| Memory cost | None extra | Queue + extra encoder weights |
| Best use case | When you have 8+ GPUs | When you have 1–2 GPUs |
| Linear probe (ResNet-50, 200 ep) | 66.5% | 67.5% (MoCo v2) |

The two are *equivalent* in their loss function — both use InfoNCE. The difference is engineering. MoCo's queue + EMA is a more economical way to get the same statistical effect.

### 11.5 MoCo queue + momentum, end to end


In [ ]:
# ── MoCo: momentum encoder + queue ────────────────────────────────────────
class MoCo(nn.Module):
    def __init__(self, encoder_ctor, dim=128, K=4096, m=0.999, tau=0.07):
        super().__init__()
        self.K, self.m, self.tau = K, m, tau
        self.encoder_q = encoder_ctor(dim)
        self.encoder_k = encoder_ctor(dim)

        # Initialise key encoder = query encoder, then freeze gradients
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data.copy_(pq.data)
            pk.requires_grad = False

        # Queue of negatives
        self.register_buffer("queue", F.normalize(torch.randn(dim, K), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update(self):
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data.mul_(self.m).add_(pq.data, alpha=1 - self.m)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        B = keys.size(0)
        ptr = int(self.queue_ptr)
        # Assume B divides K for simplicity
        self.queue[:, ptr:ptr + B] = keys.t()
        self.queue_ptr[0] = (ptr + B) % self.K

    def forward(self, x_q, x_k):
        q = F.normalize(self.encoder_q(x_q), dim=-1)        # gradients flow

        with torch.no_grad():
            self._momentum_update()
            k = F.normalize(self.encoder_k(x_k), dim=-1)    # no gradients

        # Positive logits: (B, 1)
        l_pos = (q * k).sum(-1, keepdim=True)
        # Negative logits: (B, K)
        l_neg = q @ self.queue.clone().detach()
        logits = torch.cat([l_pos, l_neg], dim=1) / self.tau

        labels = torch.zeros(q.size(0), dtype=torch.long, device=q.device)
        loss = F.cross_entropy(logits, labels)

        self._dequeue_and_enqueue(k)
        return loss

# Tiny smoke test with a toy encoder
def tiny_enc(dim):
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, dim))

moco = MoCo(tiny_enc, dim=64, K=512, m=0.999).to(device)
opt = torch.optim.SGD(moco.encoder_q.parameters(), lr=0.03, momentum=0.9)

for step in range(3):
    x, _ = next(iter(loader))
    x = x.to(device)
    # Two views = two random Gaussian augmentations of the same image
    xq = x + 0.05 * torch.randn_like(x)
    xk = x + 0.05 * torch.randn_like(x)
    loss = moco(xq, xk)
    opt.zero_grad(); loss.backward(); opt.step()
    print(f"step {step}: loss = {loss.item():.4f}, queue_ptr = {int(moco.queue_ptr)}")


---

# Part IV — Non-Contrastive Learning

## Chapter 12 — The collapse problem

*(Lecture slides 32–33)*

### 12.1 What if we removed the negatives?

Contrastive learning has two forces in the loss:

- **Attraction**: $-\,q^\top k^+$ — pulls the positive together.
- **Repulsion**: the $\sum_j \exp(q^\top k^-_j / \tau)$ term — pushes the negatives apart.

What if we kept only the attractive part? Define the naïve "non-contrastive" loss

$$\mathcal{L}_{\text{naïve}} = -\,\mathrm{sim}(z_1, z_2)$$

where $z_1$ and $z_2$ are embeddings of two views of the same image.

This loss has a global minimum: **map every input to the same constant vector $c$.** Then $\mathrm{sim}(c, c) = 1$ for all inputs, the loss is at its minimum value $-1$, and the network has learned **nothing**. Every image has the same representation.

This is **representation collapse**. It is the central problem of non-contrastive learning, and it is both a theoretical inevitability for $\mathcal{L}_{\text{naïve}}$ and a practical disaster — collapse in training looks like normal loss curves, but downstream linear-probe accuracy drops to chance.

### 12.2 Why negatives prevented collapse

In contrastive learning, the constant solution does *not* minimise the loss. If every $q$ and every $k_j$ are the same constant $c$, then in InfoNCE:

$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{e^{1/\tau}}{(K + 1) e^{1/\tau}} = \log(K + 1).$$

That is the **maximum** of the loss, not the minimum. The repulsion term explicitly punishes solutions where all embeddings are the same. **Negatives are an anti-collapse mechanism, not just a source of difficulty.**

### 12.3 Why does anyone want to remove them anyway?

Because negatives are expensive (huge batches or queues) and they create their own problems:

- **False negatives**: two different images of the same class get pushed apart by InfoNCE. The network learns *instance discrimination* rather than *class discrimination* — the augmentation distribution decides whether this is a feature or a bug.
- **Compute**: the $O(N^2)$ similarity matrix at each step.
- **Hyper-parameter sensitivity**: temperature, batch size, number of negatives, queue length.

If we could replace the repulsive force with a structural constraint that makes collapse impossible *by construction*, we'd get a simpler method. That is the agenda of BYOL, SimSiam, and DINO.


## Chapter 13 — The asymmetry principle

*(Lecture slides 34–35)*

### 13.1 The unifying observation

After three years of empirical work and a great deal of confusion, the field converged on one organising principle:

> **Non-contrastive methods avoid collapse by introducing *asymmetry* between the two branches of the Siamese network.**

The asymmetry can take one of three forms:

1. **Asymmetric architecture** — one branch has an extra component the other doesn't. (BYOL and SimSiam: a *predictor* MLP on one branch only.)
2. **Asymmetric optimisation** — one branch receives gradients, the other doesn't, *or* one updates fast and the other slowly. (Stop-gradient and EMA target networks.)
3. **Asymmetric targets** — the target distribution is post-processed (centred, sharpened, multi-cropped) before being used. (DINO.)

### 13.2 Why asymmetry blocks collapse — the intuition

Suppose both branches are *identical* and we apply the loss $-\mathrm{sim}(z_1, z_2)$. Both branches collaborate to find the easiest minimum, and the easiest minimum is "everyone is the same constant". There is no force inside the network that prefers a non-constant solution.

Now break the symmetry. One branch has to **predict** what the other branch outputs, and the other branch is treated as a **fixed target** (stop-gradient). The predictor must do *non-trivial work*: it must extract from one view enough information to predict the embedding of the other view. The fixed target is not "cooperating" — it has been frozen, so the predictor cannot lobby it to also collapse.

The mathematics of why exactly this works is still partially open (see Tian et al. NeurIPS 2021 and Wen & Li ICLR 2022 for partial theories), but the empirical fact is unambiguous: **whenever you remove the predictor or the stop-gradient, the model collapses immediately**. We will see this experimentally in Chapter 15.

### 13.3 The three implementations

| Method | Architectural asymmetry | Optimisation asymmetry | Target asymmetry |
|---|---|---|---|
| BYOL    | predictor on online branch | stop-grad + EMA target | — |
| SimSiam | predictor on one branch | stop-grad only | — |
| DINO    | (none — both branches same) | EMA teacher | centring + sharpening + multi-crop |

The trend going down the table is *removing* mechanisms while keeping the model from collapsing. SimSiam is "BYOL minus EMA". DINO trades the architectural asymmetry for a target-distribution asymmetry. All three are stable; none uses negatives.


## Chapter 14 — BYOL: Bootstrap Your Own Latent

*(Lecture slides 36–40; Grill et al., NeurIPS 2020)*

### 14.1 Architecture

BYOL has two networks that share the same architecture but **not** the same weights:

- **Online network** (gradient-trained): encoder $f_\theta$ → projector $g_\theta$ → **predictor** $q_\theta$.
- **Target network** (no gradients): encoder $f_\xi$ → projector $g_\xi$. **No predictor.**

The target network's parameters $\xi$ are an exponential moving average of $\theta$:

$$\xi \leftarrow \tau\, \xi + (1 - \tau)\, \theta.$$

For each image we draw two augmented views $v$, $v'$. The forward pass:

1. Online: $v \to f_\theta \to g_\theta \to q_\theta \to p_\theta(v)$.
2. Target: $v' \to f_\xi \to g_\xi \to z'_\xi(v')$.   *(no predictor!)*

The loss is the cosine distance between the online prediction and the target projection — and, crucially, with **stop-gradient** on the target side:

$$\mathcal{L}_{\text{BYOL}} = 2 - 2\,\frac{\langle p_\theta(v),\, \mathrm{sg}(z'_\xi(v'))\rangle}{\|p_\theta(v)\|\,\|z'_\xi(v')\|}.$$

The loss is then symmetrised by swapping $v$ and $v'$.

### 14.2 Why doesn't BYOL collapse?

This is the question the BYOL paper *did not answer cleanly*, and the lecture is honest about that: **there is no clean theorem.** The empirical answer has three parts:

1. **The predictor is the single most important component.** Removing the predictor produces immediate collapse. The predictor introduces *architectural asymmetry* — only the online branch has it, so the online branch must do non-trivial work to match the target.
2. **The EMA target is a moving target.** Even if the online branch is tempted to collapse, the target it must match is changing slowly. There is no fixed-point trivial solution to converge to.
3. **Stop-gradient blocks co-adaptation.** Without stop-gradient, both branches would receive gradients and could *cooperate* to find a collapsed solution. Stop-gradient turns the target into a fixed regression target.

A useful slogan: **BYOL learns by predicting a slowly moving past version of itself.**

### 14.3 Empirical results (the conceptual breakthrough)

| Method | Linear probe top-1 (ImageNet, ResNet-50) |
|---|---|
| Supervised baseline | 76.5% |
| SimCLR | 69.3% |
| MoCo v2 | 71.1% |
| **BYOL** | **74.3%** |
| BYOL + ResNet-200 (2×) | 79.6% |

The numbers matter, but the *conceptual* result matters more: **BYOL beats every contrastive method using zero negatives.** Before BYOL, the field believed that negatives were essential. BYOL was the proof-of-existence that they are not. SimSiam and DINO followed within months.

### 14.4 Implementation checklist

The BYOL paper hides several non-obvious recipe details. For self-study, this is the list to know.

| Component | Setting | Why |
|---|---|---|
| EMA coefficient $\tau$ | Cosine schedule from 0.996 → 1.0 | Too low → target moves too fast, instability. Too high (e.g. 1.0 from start) → target frozen, never learns. |
| Predictor MLP | 2 layers, 4096 hidden, 256 output | Removing the predictor causes immediate collapse. **Most critical component.** |
| Augmentations | Crop + colour jitter + blur + solarise | Without strong augmentation, "alignment" is trivial. |
| Optimiser | LARS-SGD, batch 4096, 1000 epochs | Same scale as SimCLR, but the per-step compute is cheaper. |
| Larger backbones | Help BYOL more than they help contrastive | One of BYOL's most attractive scaling properties. |

The take-home: BYOL works, but it is a *recipe*, and changing any one component can break it. Read the original paper's appendix before reproducing.


## Chapter 15 — SimSiam: Simplicity is All You Need

*(Lecture slides 41–44; Chen & He, CVPR 2021)*

### 15.1 The minimal recipe

If BYOL was "negatives are unnecessary", SimSiam is "the EMA target is also unnecessary". SimSiam removes the second-most-loved component of BYOL and keeps working. The architecture:

- A **single** encoder $f$ (no separate target network).
- A **single** projector $g$ on top of $f$.
- A **predictor** $h$ on top of $g$ — only on one of the two branches.
- **Stop-gradient** applied to the *other* branch.
- No EMA. No target network. No queue. No negatives.

For two augmented views $x_1, x_2$:

$$
z_i = g(f(x_i)),\qquad
p_i = h(z_i),\qquad
\mathcal{L}_{\text{SimSiam}} = -\frac{1}{2}\!\left[\mathrm{cos}(p_1, \mathrm{sg}(z_2)) + \mathrm{cos}(p_2, \mathrm{sg}(z_1))\right].
$$

That is the entire method. There are five lines of code worth of difference between SimSiam and "the trivial collapsing baseline", and that difference is `stop_gradient` and the predictor.

### 15.2 What stop-gradient actually does

The Chen & He paper provides perhaps the cleanest empirical story in the entire SSL literature: a single ablation that toggles `stop_gradient` on/off and shows that without it, training collapses *immediately and completely*; with it, training proceeds normally.

Why? The authors propose an **alternating-optimisation** (EM-like) interpretation:

1. With stop-gradient on branch 2, the predictor on branch 1 sees branch 2's output as a *fixed* target. It updates to *predict* that target.
2. Then on the next batch (or on the symmetric loss term), the roles swap. Branch 1's output is now treated as the fixed target, and branch 2's predictor updates.

Think of it as two players playing "predict me" — but they take turns being the prediction target, and neither can lobby the other to take the easy way out (constant output) because each one is, in turn, frozen.

A second necessary ingredient is the **predictor** $h$ itself. If you remove the predictor *even with* stop-gradient, the model still collapses — the predictor's job is to do non-trivial work to translate one view's representation into the other's.

### 15.3 BYOL vs SimSiam: what is essential?

| Component | BYOL | SimSiam | Necessary? |
|---|---|---|---|
| Predictor | ✅ | ✅ | **Yes** — removing causes collapse |
| Stop-gradient | ✅ | ✅ | **Yes** — removing causes collapse |
| EMA target encoder | ✅ | ❌ | No — SimSiam works without it (slightly worse) |
| Negatives | ❌ | ❌ | No |
| Memory bank | ❌ | ❌ | No |

The minimal recipe is therefore **shared encoder + projector + predictor + stop-gradient**. Everything else is optional. SimSiam was a paper whose contribution was to show how much you could remove from BYOL without breaking it.

### 15.4 The collapse demo


In [ ]:
# ── SimSiam-style collapse demo on a synthetic 2D dataset ──────────────────
torch.manual_seed(0)

# Toy data: 256 points on an annulus
N = 256
theta = torch.rand(N) * 2 * math.pi
data = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)

class Branch(nn.Module):
    def __init__(self, d=2, h=64):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, d))
        self.predictor = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, d))
    def forward(self, x):
        z = self.encoder(x)
        p = self.predictor(z)
        return z, p

def train_simsiam(use_stop_grad: bool, epochs=300):
    net = Branch()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    losses, std_z = [], []
    for _ in range(epochs):
        # Two augmented views = small Gaussian noise
        v1 = data + 0.05 * torch.randn_like(data)
        v2 = data + 0.05 * torch.randn_like(data)
        z1, p1 = net(v1)
        z2, p2 = net(v2)
        if use_stop_grad:
            tgt1, tgt2 = z2.detach(), z1.detach()
        else:
            tgt1, tgt2 = z2, z1
        # Symmetric negative cosine
        loss = -(F.cosine_similarity(p1, tgt1).mean() +
                 F.cosine_similarity(p2, tgt2).mean()) / 2
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        std_z.append(F.normalize(z1, dim=-1).std(0).mean().item())
    return losses, std_z

losses_with,  std_with  = train_simsiam(True)
losses_no,    std_no    = train_simsiam(False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses_with, label="with stop-grad")
axes[0].plot(losses_no,   label="no stop-grad")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("step")
axes[1].plot(std_with, label="with stop-grad")
axes[1].plot(std_no,   label="no stop-grad")
axes[1].set_title("Embedding std (collapse if → 0)")
axes[1].set_xlabel("step"); axes[1].legend()
plt.tight_layout(); plt.show()

print("Final embedding std:")
print(f"  with stop-grad : {std_with[-1]:.4f}")
print(f"  no   stop-grad : {std_no[-1]:.4f}   ← ≈ 0 means collapse")


## Chapter 16 — DINO: self-distillation with no labels

*(Lecture slides 45–50; Caron et al., ICCV 2021)*

### 16.1 From vector matching to distribution matching

BYOL and SimSiam align two embedding vectors with cosine similarity. DINO does something subtly different and conceptually richer: it aligns two **probability distributions** over a fixed set of latent prototypes.

Why is this an improvement? Vector matching gives the model one constraint per dimension. Distribution matching (i.e. cross-entropy over $K$ prototypes) gives the model *structural* constraints — the relative weights across all $K$ prototypes must agree across views. This is closer to "the same object" than to "the same vector".

### 16.2 The architecture

DINO has a **student** network and a **teacher** network. Both have the same architecture (typically ViT-S or ViT-B). The teacher is an EMA of the student — exactly like MoCo or BYOL — and receives no gradients.

For each input image $x$, augmentation produces:

- 2 **global** crops at $224 \times 224$ (large, mostly the whole object).
- $V$ **local** crops at $96 \times 96$ (small, possibly just a part).

| Branch | Sees |
|---|---|
| **Teacher** | Only the 2 global crops |
| **Student** | All $2 + V$ crops (globals + locals) |

The student's job: even from a small local crop, produce a distribution that matches the teacher's distribution from the global crop. The local-to-global asymmetry is what teaches the student that *partial views must be consistent with the whole*.

### 16.3 Softmax + temperature on both sides

Both networks output a $K$-dimensional vector (e.g. $K = 65536$). They convert to probabilities via softmax with **different temperatures**:

$$
P_s(x) = \mathrm{softmax}\!\left(\frac{g_s(x)}{\tau_s}\right), \qquad
P_t(x) = \mathrm{softmax}\!\left(\frac{g_t(x) - c}{\tau_t}\right).
$$

Two new ingredients here:

- **Sharpening** ($\tau_t \ll \tau_s$, e.g. $\tau_t = 0.04$, $\tau_s = 0.1$) — the teacher distribution is much *sharper* than the student. This gives the student a confident target to imitate.
- **Centring** ($-c$ on the teacher logits, where $c$ is an EMA over the batch mean) — this prevents one prototype from dominating across all images.

### 16.4 Centring vs sharpening: a tug-of-war

Without intervention, two distinct collapse modes are possible:

- **Mode collapse** — every image picks the same prototype. The teacher distribution is "everyone is in cluster #42".
- **Uniform collapse** — every image gets a uniform distribution. The teacher distribution is maximum-entropy and contains no information.

DINO walks a tightrope between them:

- **Centring** subtracts the running mean from the teacher's logits. This *prevents mode collapse* — no single prototype can dominate, because subtracting its mean drags it back down.
- **Sharpening** (low $\tau_t$) *prevents uniform collapse* — the teacher's softmax is concentrated, so the targets are confident.

Both are necessary. Centring alone → uniform collapse. Sharpening alone → mode collapse. Together → a healthy distribution that uses many prototypes without being uniform.

### 16.5 Multi-crop training

A clever data-augmentation trick. The student processes both 2 globals and $V$ locals; the teacher processes only the 2 globals. Loss:

$$
\mathcal{L}_{\text{DINO}} = \sum_{x_g \in \text{globals}}\;\sum_{x \neq x_g}\; H\!\left(P_t(x_g),\, P_s(x)\right)
$$

where $H(p, q) = -\sum_i p_i \log q_i$ is cross-entropy and the inner sum runs over all student crops *except* the matching global. The student must produce, even from small local crops, a distribution that matches the teacher's distribution on a global view.

This is where DINO gets a lot of its power. The local-to-global pressure is a strong inductive bias: *partial observations of an object must be semantically consistent with the whole object*.

### 16.6 Why DINO works especially well with ViTs

This is one of the most striking results in self-supervised vision: DINO, when trained with a ViT backbone, exhibits **emergent attention maps that segment objects without any segmentation supervision**. Visualising the attention of the `[CLS]` token onto the patch tokens highlights object boundaries — dogs, cats, cars, faces — as cleanly as a supervised segmenter.

This emergence does not happen with:

- Supervised ViTs (the ImageNet label provides no spatial signal).
- DINO with a ResNet (no `[CLS]` token, no patch tokens, no spatial-attention structure).
- BYOL or SimSiam on ViT (no distribution-level constraint).

The combination that makes the magic:

1. **ViT** — patch tokens give explicit spatial positions; `[CLS]` aggregates globally.
2. **Self-distillation** — distribution matching is a much richer constraint than vector matching.
3. **Multi-crop** — local crops force per-region semantic consistency.

This is the strongest argument that *the right architecture and the right pretext task interact*. ViT alone is not magic; DINO alone is not magic; the combination is.

### 16.7 A toy DINO centring + sharpening demonstration


In [ ]:
# ── DINO centring & sharpening: how the two prevent the two collapse modes ─
torch.manual_seed(0)

K = 8                          # 8 prototypes
B = 256                        # batch
n_iter = 200

def softmax_with(logits, temp, center=None):
    if center is not None:
        logits = logits - center
    return F.softmax(logits / temp, dim=-1)

def simulate(use_center: bool, tau_t: float):
    # Simulate teacher distributions over 8 prototypes.
    # Random "teacher logits" that drift toward concentrating on prototype 0
    torch.manual_seed(0)
    logits = torch.randn(B, K)
    bias = torch.zeros(K); bias[0] = 1.0   # prototype 0 has slight head-start
    center = torch.zeros(K)
    history_max = []
    history_entropy = []
    for it in range(n_iter):
        logits = logits + 0.05 * bias       # network drifts toward mode collapse
        p = softmax_with(logits, tau_t, center if use_center else None)
        # Update centre as EMA of batch mean
        if use_center:
            center = 0.9 * center + 0.1 * logits.mean(0)
        history_max.append(p.mean(0).max().item())  # mass on dominant prototype
        history_entropy.append(-(p * p.clamp_min(1e-9).log()).sum(-1).mean().item())
    return history_max, history_entropy

# Four settings
settings = [
    ("no center, sharp τ=0.04",  False, 0.04),
    ("center,    sharp τ=0.04",  True,  0.04),
    ("no center, soft  τ=1.00",  False, 1.00),
    ("center,    soft  τ=1.00",  True,  1.00),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, c, t in settings:
    m, h = simulate(c, t)
    axes[0].plot(m, label=label)
    axes[1].plot(h, label=label)
axes[0].set_title("Mass on dominant prototype  →  1.0 means MODE collapse")
axes[0].set_ylim(0, 1); axes[0].set_xlabel("step"); axes[0].legend(fontsize=8)
axes[1].set_title("Entropy  →  log(8)≈2.08 means UNIFORM collapse")
axes[1].axhline(math.log(K), ls="--", c="grey", label="log K (uniform)")
axes[1].set_xlabel("step"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


**Reading the figure.**

- *No centre + sharp τ* — drifts to mode collapse (one prototype gets all the mass).
- *No centre + soft τ* — uniform: high entropy, no information.
- *Centre + sharp τ* — DINO's regime: one prototype is preferred, but centring keeps it from dominating; sharp τ keeps the distribution concentrated. **A healthy non-trivial distribution.**
- *Centre + soft τ* — the centre fights mode collapse but sharp τ is missing, so we are pulled back toward uniform.

Both ingredients are necessary. This is the cleanest 1-screen demo of the DINO design.


---

# Part V — Multimodal Self-Supervision

## Chapter 17 — CLIP through the SSL lens

*(Lecture slides 51–55; Radford et al., ICML 2021)*

### 17.1 The pivot: where do positives come from?

In SimCLR, a positive pair is $(t(x), t'(x))$ — two augmented views of the *same image*. The augmentation distribution defines what should be invariant.

In CLIP, a positive pair is $(\text{image}, \text{caption})$ — an image and a *natural-language description* of it, scraped from the internet. The supervisor is the human who wrote the alt-text. The "augmentation" that defines invariance is now: **everything two different captions of the same scene have in common.**

This is a much richer signal than crop+colour-jitter, because language already abstracts away the irrelevant. A caption "a black dog on grass" doesn't mention the lighting, the angle, or the weather, so the model trained to align the image with that caption must learn to ignore those things on its own.

### 17.2 CLIP = cross-modal InfoNCE

The architecture has two encoders:

- **Image encoder** $f_I$ — a ResNet-50 or ViT-L/14. Output: $I_i \in \mathbb{R}^{512}$.
- **Text encoder** $f_T$ — a Transformer (63M params). Output: $T_i \in \mathbb{R}^{512}$.

Both outputs are L2-normalised and live in the same shared 512-d space. For a batch of $N$ image-text pairs $\{(x_i, c_i)\}_{i=1}^{N}$, define the similarity matrix

$$S_{ij} = (I_i)^\top T_j / \tau.$$

The diagonal entries are positives (image $i$ matches caption $i$); off-diagonal entries are negatives. The CLIP loss is **symmetric InfoNCE**:

$$
\mathcal{L}_{\text{CLIP}} = -\frac{1}{2N}\sum_{i=1}^{N}\!\left[\,
\log\frac{e^{S_{ii}}}{\sum_j e^{S_{ij}}}
+ \log\frac{e^{S_{ii}}}{\sum_j e^{S_{ji}}}
\,\right].
$$

The first term is "from each image, find the matching caption". The second term is "from each caption, find the matching image". Both are exactly the SimCLR loss — it is the same NT-Xent — applied to a similarity matrix that is *cross-modal* rather than *within-image*.

> CLIP is structurally identical to SimCLR. The only difference is the source of the positive pair.

### 17.3 Training scale

The numbers are part of the story, because CLIP only works at scale:

| Resource | CLIP (ViT-L/14) |
|---|---|
| Training pairs | 400M (image, alt-text) from the web (the WIT dataset) |
| Batch size | 32,768 |
| Negatives per anchor | 32,767 (in-batch) |
| Hardware | 256 V100 GPUs |
| Training time | 12 days |
| $\tau$ | Learned, initialised so $\log(1/\tau) \approx 2.6$, clipped to $\le \ln 100$ |

The batch size is the highest of any contrastive method we have seen, because more in-batch negatives = better InfoNCE bound = better representation. Cross-modal alignment is the ultimate "more negatives = better" workload.

### 17.4 Cross-modal contrastive loss from scratch


In [ ]:
# ── CLIP loss = symmetric NT-Xent on a cross-modal similarity matrix ──────
def clip_loss(image_feats, text_feats, log_tau):
    # image_feats, text_feats : (N, d) -- already L2-normalised.
    # log_tau : scalar temperature parameter (CLIP learns it).
    tau = log_tau.exp()
    logits_per_image = image_feats @ text_feats.t() * tau    # (N, N)
    logits_per_text  = logits_per_image.t()
    targets = torch.arange(image_feats.size(0), device=image_feats.device)
    li = F.cross_entropy(logits_per_image, targets)
    lt = F.cross_entropy(logits_per_text,  targets)
    return (li + lt) / 2

# Toy example: N=8 fake (image, caption) pairs
N, d = 8, 64
img = F.normalize(torch.randn(N, d), dim=-1)
txt = F.normalize(img + 0.05 * torch.randn(N, d), dim=-1)   # well-aligned pairs
log_tau = torch.tensor(2.6, requires_grad=True)             # CLIP init

print(f"CLIP loss (well-aligned pairs)   = {clip_loss(img, txt, log_tau).item():.4f}")
txt_random = F.normalize(torch.randn(N, d), dim=-1)
print(f"CLIP loss (random pairs)         = {clip_loss(img, txt_random, log_tau).item():.4f}")
print(f"Random-baseline log(N)           = {math.log(N):.4f}")


## Chapter 18 — Why CLIP matters, and where it breaks

*(Lecture slides 56–57)*

### 18.1 Why CLIP changed the field

| # | Change | Why it matters |
|---|---|---|
| **1. Free supervision at internet scale** | 400M (image, alt-text) pairs cost zero to collect. The same pipeline can be scaled to **5B+** (LAION-5B). | The data ceiling on visual representation learning effectively disappeared. |
| **2. Zero-shot transfer** | "A photo of a {class}" gives a classifier for any new dataset without a single labelled example. | CLIP's zero-shot ImageNet (76.2%) matches a ResNet-50 *trained* on ImageNet. |
| **3. Robustness to distribution shift** | CLIP closes 75% of the ImageNet → ImageNet-Sketch / ObjectNet / ImageNet-A / ImageNet-R robustness gap. | The first method to *not* overfit to the standard benchmark. |
| **4. Bidirectional retrieval** | Text→image and image→text retrieval in the same embedding space. | Image search engines, dataset filtering, captioning pipelines. |
| **5. Foundation-model precursor** | Vision encoder used by DALL·E 2, Stable Diffusion, GPT-4V, LLaVA, IDEFICS, … | Almost every multimodal model since 2021 either *is* CLIP or *uses* CLIP. |

### 18.2 Where CLIP breaks

CLIP also has clear failure modes that you must know about — these come up in any practical project that uses a CLIP encoder.

**1. Prompt sensitivity.** "A photo of a {class}" vs. "{class}" can differ by 10+ accuracy points. The exact wording matters more than it should. *Prompt engineering is a skill, not a footnote.*

**2. Data bias.** Internet alt-text is biased — gender, race, age, geography. CLIP encodes those biases, and worse, *concentrates* them in a shared embedding space where they are hard to disentangle. The original paper has an honest discussion of this in section 7.

**3. Weak compositional understanding.** CLIP matches **global** image-caption similarity. It is bad at:
- *Counting*: "three dogs" vs "two dogs" — CLIP can barely tell them apart.
- *Spatial relations*: "the cat to the left of the dog" — order is lost.
- *Attribute binding*: "the red cube and the blue sphere" vs "the blue cube and the red sphere" — colours and objects get swapped.

This is because the loss only constrains the *whole* image to match the *whole* caption — it never asks the model to bind specific words to specific image regions.

**4. Language quality is the ceiling.** If alt-text is noisy or shallow, the model learns shallow features. Better-curated text data → better representations. This is why LAION, DataComp, CC12M, WebLI, etc. all matter — they are attempts to clean up the supervision.

### 18.3 The "semantic alignment" framing

Lecture slide 58 offers the most compact way to see what CLIP did to the field:

> **If SimCLR learns invariances, and DINO learns structure, then CLIP learns semantics.**

| Method | What is preserved | What is the supervisor |
|---|---|---|
| **SimCLR** | Invariance to a fixed augmentation set | The augmentation distribution chosen by us |
| **DINO** | Distributional structure over latent prototypes | The teacher network (an EMA of the student) |
| **CLIP** | The semantic meaning of natural language | Human-written captions on the open web |

CLIP is the first method whose supervision signal is itself *pre-loaded with semantics*. SimCLR has to discover from raw pixels that "rotated dog ≈ unrotated dog"; CLIP gets that for free, because the caption already says "dog" regardless of rotation. This is why CLIP is the bridge between perception (representation learning) and language — it turns representation learning into a *semantic alignment* problem.

This bridge is the reason every modern multimodal model is built on or in response to CLIP. The next generation — BLIP, ALIGN, Florence, SigLIP, EVA-CLIP — all play in this design space.


## Chapter 19 — Three philosophies on one slide

*(Lecture slide 58)*

A useful table to keep in your head. All three methods are "Siamese-style" representation learners, and yet each makes a different choice about what *should* be the same across the two views.

| | What is the same? | Where the supervision lives | What you get |
|---|---|---|---|
| **SimCLR / MoCo** | Two augmentations of the **same image** | The augmentation distribution | Augmentation invariance, instance discrimination |
| **DINO** | Two crops of the **same image**, but matched as **distributions** over prototypes | The teacher network (EMA) | Distributional structure, emergent segmentation |
| **CLIP** | An **image** and its **caption** | Human-written language on the web | Semantic alignment between vision and language |

Each row is a different answer to: *what counts as "the same"?* And each row gives the model a different inductive bias:

- SimCLR's bias: things that look similar after random crop + colour jitter are conceptually similar.
- DINO's bias: small parts of an object should be probabilistically consistent with the whole object.
- CLIP's bias: things that humans describe with similar words *are* similar.

The progression — from augmentation, to multi-crop distribution, to natural language — is also a progression in **how much human knowledge is baked into the supervision signal**. SimCLR's only human input is the augmentation list. DINO adds an architectural assumption about local-global structure. CLIP adds the entire English language.

This is the trajectory of representation learning over the last five years: the supervision signal has become richer, more structured, and more informationally aligned with what we ultimately want — semantic understanding.


---

# Part VI — Synthesis

## Chapter 20 — Final takeaways

*(Lecture slide 59)*

If you remember nothing else from this notebook, remember the following five points.

### 20.1 Every SSL method designs a supervision signal

There is no such thing as "label-free learning" — there are only methods that **manufacture labels from the data**. Each method is best understood by asking: *what is the network trying to predict, and from what?*

| Method | Signal type | What is predicted? | From what? |
|---|---|---|---|
| AE / DAE | Reconstruction | Input pixels | Bottlenecked input |
| MAE | Reconstruction | Masked patches | Visible patches |
| SimCLR / MoCo | Discrimination | Which other view is the positive | Anchor view + many negatives |
| BYOL / SimSiam | Alignment | Target embedding | Online embedding |
| DINO | Distribution alignment | Teacher distribution | Student distribution |
| CLIP | Cross-modal alignment | Caption embedding | Image embedding |

### 20.2 *"What should be the same?"* is the central design choice

This is the most important question in representation learning, and every method gives a different answer:

- **AE / MAE**: The reconstruction should be the same as the input.
- **SimCLR**: Two augmented views of the same image should be the same.
- **DINO**: A small patch should give the same *distribution* over prototypes as the whole image.
- **CLIP**: An image should be the same as its caption (in embedding space).

Choose well and you get rich, transferable features. Choose poorly and you get a network that memorises shortcuts. *The art of representation learning is the art of choosing the right invariance.*

### 20.3 For each method, know its four pieces

When you read a new SSL paper, look for these four components:

1. **The objective function.** What is the loss?
2. **The target.** What does the loss compare against — pixels, an embedding, a distribution?
3. **The anti-collapse mechanism.** If we removed it, would the network collapse to a constant?
4. **The evaluation protocol.** Linear probe? Fine-tune? Zero-shot? Each protocol measures a slightly different thing.

If you cannot answer all four for a method, you do not yet understand it.

### 20.4 Representation learning = choosing the right invariance

Too much invariance and you destroy useful information ("all dogs are the same"). Too little and the network learns shortcuts ("colour histograms are enough"). The right amount is task-dependent. Some empirical rules:

- **For classification**: aggressive invariance is good. SimCLR / DINO style augmentations.
- **For detection / segmentation**: spatial information must be preserved. MAE-style methods often beat contrastive ones.
- **For retrieval**: the embedding geometry matters. CLIP-style cross-modal alignment is the gold standard.
- **For fine-grained classification (birds, cars)**: be careful — colour distortion can erase the very signal you need.

### 20.5 Asymmetry prevents collapse in non-contrastive methods

This is the single deepest unifying principle from Part IV. Non-contrastive methods avoid collapse not by punishing the constant solution explicitly (negatives), but by introducing a **structural asymmetry** that makes the constant solution unreachable.

The asymmetry can be:

- **Architectural**: a predictor MLP on one branch only (BYOL, SimSiam).
- **Optimisational**: stop-gradient + EMA (BYOL, DINO).
- **Distributional**: centring + sharpening + multi-crop (DINO).

All three forms achieve the same effect — they break the symmetry of "branch 1 = branch 2" enough that the trivial collapsing solution is no longer a fixed point of training.

---

### A final word

Self-supervised learning is now the default way to pre-train large vision and language models. Every foundation model you have heard of — GPT-4, Gemini, Claude, Llama, Stable Diffusion, DALL·E, SAM, DINOv2 — is built on a self-supervised pre-training stage. The methods in this notebook are the *reasoning patterns* behind those systems.

Of the methods we covered:

- **MAE** lives on inside DINOv2 (which combines MAE-style and DINO-style losses).
- **CLIP** lives on inside every vision-language model.
- **MoCo** and **BYOL** ideas live on inside every modern training recipe (EMA target, queues for negatives, predictor heads).
- **SimSiam** is rarely used directly but its conceptual contribution — that a single line of code (`stop_gradient`) is enough to prevent collapse — reframed how the field thinks about Siamese training.

Understanding these methods is therefore not just a matter of historical interest. The vocabulary of *positive pairs, negatives, predictors, EMAs, centring, sharpening, multi-crop, augmentation invariance, instance discrimination, distribution alignment* is the vocabulary in which every new representation-learning paper is written. From here you are ready to read DINOv2, BEiT, SigLIP, MaskFeat, and whatever the next breakthrough turns out to be.

---

## References and further reading

- Hinton & Salakhutdinov, *Reducing the Dimensionality of Data with Neural Networks*, **Science 2006** — the autoencoder paper.
- Vincent et al., *Extracting and Composing Robust Features with Denoising Autoencoders*, **ICML 2008**.
- He et al., *Masked Autoencoders Are Scalable Vision Learners*, **CVPR 2022**.
- van den Oord, Li, Vinyals, *Representation Learning with Contrastive Predictive Coding*, **arXiv 2018** — InfoNCE.
- Chen, Kornblith, Norouzi, Hinton, *A Simple Framework for Contrastive Learning of Visual Representations* (SimCLR), **ICML 2020**.
- He et al., *Momentum Contrast for Unsupervised Visual Representation Learning* (MoCo), **CVPR 2020**.
- Grill et al., *Bootstrap Your Own Latent* (BYOL), **NeurIPS 2020**.
- Chen & He, *Exploring Simple Siamese Representation Learning* (SimSiam), **CVPR 2021**.
- Caron et al., *Emerging Properties in Self-Supervised Vision Transformers* (DINO), **ICCV 2021**.
- Radford et al., *Learning Transferable Visual Models From Natural Language Supervision* (CLIP), **ICML 2021**.
- Wang & Isola, *Understanding Contrastive Representation Learning through Alignment and Uniformity on the Hypersphere*, **ICML 2020** — the alignment/uniformity framework.
- Tian, Chen, Ganguli, *Understanding Self-supervised Learning Dynamics without Contrastive Pairs*, **ICML 2021** — partial theory of why SimSiam doesn't collapse.
- Oquab et al., *DINOv2: Learning Robust Visual Features without Supervision*, **TMLR 2024** — the modern descendant that combines many of the methods in this notebook.
